# Target Encoder — Interview Revision Notes

## 1. What is Target Encoding?

**Target Encoding** is an encoding technique that converts a categorical variable into a numerical value based on the **target variable**.

For a binary classification problem:

$$
TE(c) = P(y=1 \mid X=c)
$$

In simple terms:

> Replace each category with the mean of the target values observed for that category.

### Example

Suppose:

| City | Purchased |
|---|---:|
| Delhi | 1 |
| Delhi | 1 |
| Delhi | 0 |
| Mumbai | 0 |
| Mumbai | 1 |
| Raipur | 0 |

Then:

$$
TE(Delhi)=\frac{1+1+0}{3}=0.67
$$

$$
TE(Mumbai)=\frac{0+1}{2}=0.50
$$

$$
TE(Raipur)=0
$$

---

## 2. The Problem: Rare Categories

The problem occurs when a category has **very few observations**.

Suppose:

```text
Category A → 1000 observations → target mean = 0.62

Category B → 2 observations → target mean = 1.00

Can we trust the 1.00 for Category B?

Probably not.

Those two observations may simply be due to chance.

If we directly use the category mean, the model may treat:

Category B → 1.00

as very strong evidence.

This can cause overfitting.

Therefore, we need regularization / smoothing.
```

## 3. Basic Solution: Combine Category Mean and Global Mean

Instead of completely trusting the category mean, combine:

- Category mean → what we observed for this category
- Global mean → what we observe across the entire dataset

General form:

$$
TE(c) = \lambda_c \cdot CategoryMean_c + (1-\lambda_c)\cdot GlobalMean
$$

where:

$$
0 \leq \lambda_c \leq 1
$$

### Interpretation

If:

$$
\lambda_c \approx 1
$$

we trust the category mean.

If:

$$
\lambda_c \approx 0
$$

we trust the global mean.

Therefore, the central question becomes:

How should we determine $\lambda_c$?

## 4. Lambda Should Depend on the Number of Observations

Let:

$$
n_c = \text{number of observations belonging to category } c
$$

Intuitively:

- Small $n_c$ → less evidence → trust category mean less → $\lambda$ closer to 0 → global mean gets more weight
- Large $n_c$ → more evidence → trust category mean more → $\lambda$ closer to 1 → category mean gets more weight

Therefore:

$$
\lambda_c = f(n_c)
$$

We want a function that maps the number of observations to a value between 0 and 1.

## 5. Why Do We Use a Sigmoid Function?

The sigmoid function is:

$$
\sigma(x)=\frac{1}{1+e^{-x}}
$$

Its output is always between:

$$
0 < \sigma(x) < 1
$$

Therefore, it is useful for constructing our weighting factor $\lambda$.

A common formulation used by category_encoders is:

$$
\boxed{ \lambda_c = \sigma\left( \frac{n_c-k}{s} \right) }
$$

where:

- $n_c$ = number of observations in category $c$
- $k$ = min_samples_leaf
- $s$ = smoothing
- $\sigma$ = sigmoid function

Therefore:

$$
TE(c) = \lambda_c CategoryMean_c + (1-\lambda_c)GlobalMean
$$

## 6. Understanding the Sigmoid Curve

The sigmoid gives us a smooth transition:

```text
λ
1 |                         ______
  |                      __/
  |                   __/
0.5|------------------●
  |                __/
  |             __/
0 |____________/
  |
  +-------------------------------- n
                   ↑
             min_samples_leaf
```

The important point is:

$$
n_c = k \Rightarrow \lambda_c = 0.5
$$

because:

$$
\sigma(0)=0.5
$$

So min_samples_leaf determines the point where the encoder gives equal weight to:

- Category mean
- Global mean

## 7. min_samples_leaf (Trust Factor)

min_samples_leaf controls where the sigmoid curve is centered.

It represents the amount of evidence we want before the category mean starts receiving substantial weight.

### Example

Suppose:

min_samples_leaf = 20

Then:

$$
n=20 \Rightarrow \lambda=0.5
$$

So:

- $n < 20$ → $\lambda < 0.5$
- $n = 20$ → $\lambda = 0.5$
- $n > 20$ → $\lambda > 0.5$

### Important intuition

Increasing min_samples_leaf shifts the sigmoid to the right.

```text
λ
1 |                         ______
  |                      __/
0.5|--------------------●
  |                  __/
0 |_______________/
  +-------------------------------- n
                     ↑
              larger min_samples_leaf
```

Therefore:

A larger min_samples_leaf means we require more observations before strongly trusting the category mean.

## 8. smoothing

smoothing controls the steepness / flatness of the sigmoid transition.

### Low smoothing

The transition happens quickly.

```text
λ
1 |                 ______
  |              __/
0.5|------------●
  |          __/
0 |_________/
  +---------------------- n
```

A small change in category size can cause a relatively large change in $\lambda$.

### High smoothing

The transition becomes gradual.

```text
λ
1 |                       ___
  |                   ___/
0.5|-----------------●
  |              ___/
0 |_____________/
  +---------------------- n
```

Therefore:

Higher smoothing → flatter sigmoid → more gradual transition.

This prevents an abrupt change in the amount of trust assigned to the category mean.

## 9. Important Correction About Rare Categories

It is not correct to say:

> "n = 5 always gives $\lambda \approx 0.99$."

The value of $\lambda$ depends on both:

$$
 n,\quad min\_samples\_leaf,\quad smoothing
$$

For example:

$$
\lambda = \sigma\left( \frac{5-20}{10} \right) = \sigma(-1.5) \approx 0.18
$$

So the category mean receives only about 18% weight.

The remaining:

$$
1-0.18=0.82
$$

or 82% comes from the global mean.

## 10. Complete Target Encoding Formula

The complete process is:

### Step 1: Calculate category mean

$$
CategoryMean_c = \frac{\sum y_i \text{ for category }c}{n_c}
$$

### Step 2: Calculate global mean

$$
GlobalMean = \frac{\sum y_i}{N}
$$

### Step 3: Calculate weight

For the sigmoid approach:

$$
\lambda_c = \sigma \left( \frac{n_c-k}{s} \right)
$$

### Step 4: Combine them

$$
\boxed{ TE(c)= \lambda_c CategoryMean_c + (1-\lambda_c)GlobalMean }
$$

## 11. Why Does Target Encoding Work?

Target Encoding provides a numerical representation of a category based on its relationship with the target.

For example:

| City | Target Encoding |
|---|---:|
| Delhi | 0.72 |
| Mumbai | 0.41 |
| Raipur | 0.63 |

The model can now work with numerical features instead of potentially thousands of one-hot encoded columns.

This is particularly useful for:

- High-cardinality categorical variables
- Large datasets
- Categories with meaningful target relationships

## 12. Advantages

### 1. Handles High Cardinality

Unlike One-Hot Encoding:

```text
1000 categories
    ↓
1000 features
```

Target Encoding can maintain:

```text
1000 categories
    ↓
1 encoded feature
```

for binary/continuous targets.

For multiclass classification, scikit-learn produces one encoded feature per class for each original feature.

### 2. Captures Target Relationship

The encoding contains information about:

$$
P(y|category)
$$

rather than assigning arbitrary numbers.

### 3. Can Handle Unseen Categories

A new category encountered during transformation can be mapped to the global target mean in common implementations.

For example:

Training categories:

- Delhi
- Mumbai
- Raipur

New category:

- Bangalore

Instead of failing, the encoder can use:

$$
TE(Bangalore)=GlobalMean
$$

The exact behavior depends on the implementation/settings.

### 4. Lower Dimensionality Than One-Hot Encoding

Especially useful when categorical variables have many unique categories.

## 13. Disadvantages

### 1. Target Leakage

This is the biggest concern.

Because the encoding uses:

$$
y
$$

we are using information from the target to create a feature.

If we calculate the encoding for a training observation using the target of that same observation, the encoded feature can indirectly contain information about the answer we are trying to predict.

This can make training performance look artificially high.

## 14. How Do We Prevent Target Leakage?

### Solution: Cross-Fitting / Out-of-Fold Target Encoding

Split the training data into folds.

For example:

```text
Fold 1       Fold 2       Fold 3       Fold 4       Fold 5
```

To encode Fold 1:

Use Fold 2 + Fold 3 + Fold 4 + Fold 5

```text
                ↓
       Calculate encoding
                ↓
          Encode Fold 1
```

We do not use Fold 1's target values to calculate Fold 1's encoding.

Repeat for every fold.

- Fold 1 → encoded using 2,3,4,5
- Fold 2 → encoded using 1,3,4,5
- Fold 3 → encoded using 1,2,4,5
- Fold 4 → encoded using 1,2,3,5
- Fold 5 → encoded using 1,2,3,4

This gives us an encoding for every training observation without using its own target.

## 15. scikit-learn TargetEncoder

scikit-learn provides:

```python
from sklearn.preprocessing import TargetEncoder
```

The underlying shrinkage formula is different from the sigmoid implementation used by category_encoders.

scikit-learn uses:

$$
\boxed{ \lambda_c = \frac{n_c}{n_c+m} }
$$

where:

- $n_c$ = number of observations in category $c$
- $m$ = smoothing parameter

Then:

$$
\boxed{ TE(c)= \lambda_c CategoryMean_c + (1-\lambda_c)GlobalMean }
$$

So the idea is exactly the same:

- Category mean
- Global mean
- weighted combination

Only the function used to determine the weight is different.

## 16. Why Does scikit-learn Use a Simpler Formula?

The goal is still the same:

Shrink unreliable category statistics toward the global mean.

The scikit-learn formulation:

$$
\lambda=\frac{n}{n+m}
$$

has some useful properties:

If $n$ is small:

$$
n \ll m
$$

then:

$$
\lambda \approx 0
$$

Therefore:

$$
TE \approx GlobalMean
$$

If $n$ is large:

$$
n \gg m
$$

then:

$$
\lambda \approx 1
$$

Therefore:

$$
TE \approx CategoryMean
$$

So it naturally produces the desired behavior without requiring the explicit sigmoid parameters min_samples_leaf and smoothing.

## 17. scikit-learn smooth="auto"

scikit-learn also provides:

```python
TargetEncoder(smooth="auto")
```

Instead of manually choosing $m$, scikit-learn estimates the smoothing parameter using an empirical Bayes approach.

The documentation gives:

$$
m=\frac{\sigma_i^2}{\tau^2}
$$

where the quantities represent category-level and global variance components.

Therefore:

```text
smooth="auto"
        ↓
Estimate smoothing from the data
        ↓
Determine shrinkage automatically
```

This is one reason the scikit-learn implementation can be convenient when you don't want to manually tune the smoothing factor.

## 18. Sigmoid vs scikit-learn Formula

| Approach | Weight | Main parameters | Curve |
|---|---|---|---|
| category_encoders | Sigmoid | min_samples_leaf, smoothing | S-shaped |
| scikit-learn | $n/(n+m)$ | smooth | Rational/hyperbolic |



| Property | category_encoders | scikit-learn |
|---|---|---|
| Small categories | Shrunk toward global mean | Shrunk toward global mean |
| Large categories | Category mean gets more weight | Category mean gets more weight |
| Automatic smoothing | No equivalent | Yes |
| Main idea | Regularization | Regularization |

### Important

Neither formula is universally "better."

Both are different ways of implementing the same statistical idea:

> Do not trust a category-specific target statistic too much when there is insufficient evidence.

## 19. Multiclass Classification

Target Encoding also works for multiclass classification.

Suppose:

```text
Classes = A, B, C
```

Instead of calculating one target mean, we calculate the conditional probability for each class:

$$
P(A|category),\quad P(B|category),\quad P(C|category)
$$

Then smoothing is applied to each class probability.

scikit-learn uses a one-vs-all representation and produces:

$$
n_{features} \times n_{classes}
$$

encoded features.

### 1. Binary classification is easy

Suppose:

| City | Target |
|---|---:|
| Delhi | 1 |
| Delhi | 1 |
| Delhi | 0 |

The target mean is:

$$
\frac{1+1+0}{3}=0.67
$$

And because the target is binary:

$$
\text{Mean}(y)=P(y=1)
$$

So:

$$
TE(Delhi)=P(y=1|Delhi)
$$

Perfect. One number describes the category.

### 2. Now consider multiclass

Suppose:

| City | Target |
|---|---|
| Delhi | A |
| Delhi | A |
| Delhi | B |
| Delhi | C |

What is:

$$
Mean(A,B,C)?
$$

There is no meaningful numerical mean.

We could artificially do:

- $A = 0$
- $B = 1$
- $C = 2$

and calculate:

$$
Mean = \frac{0+0+1+2}{4}=0.75
$$

But 0.75 is meaningless.

Why?

Because we have artificially introduced an ordering:

$$
A < B < C
$$

There may be absolutely no such relationship between the classes.

### 3. What we actually want

For Delhi, what we really want is:

$$
P(A|Delhi),\quad P(B|Delhi),\quad P(C|Delhi)
$$

From our example:

Delhi → A, A, B, C

we get:

$$
P(A|Delhi)=\frac{2}{4}=0.50,
\quad
P(B|Delhi)=\frac{1}{4}=0.25,
\quad
P(C|Delhi)=\frac{1}{4}=0.25
$$

So our encoding becomes:

| City | A | B | C |
|---|---:|---:|---:|
| Delhi | 0.50 | 0.25 | 0.25 |

This is the natural multiclass target encoding.

### 4. So where does OVA come in?

This is where scikit-learn's approach becomes clever.

Instead of directly saying:

> "Calculate the proportion of A, B and C."

we can convert the multiclass target into multiple binary targets.

#### For class A

Ask:

> Is the target A?

Original:    A  A  B  C  
OVA for A:   1  1  0  0

Mean:

$$
\frac{1+1+0+0}{4}=0.5
$$

Therefore:

$$
P(A|Delhi)=0.5
$$

#### For class B

Original:    A  A  B  C  
OVA for B:   0  0  1  0

Mean:

$$
\frac{0+0+1+0}{4}=0.25
$$

Therefore:

$$
P(B|Delhi)=0.25
$$

#### For class C

Original:    A  A  B  C  
OVA for C:   0  0  0  1

Mean:

$$
\frac{0+0+0+1}{4}=0.25
$$

Therefore:

$$
P(C|Delhi)=0.25
$$

And we get:

$$
Delhi \rightarrow [0.50, 0.25, 0.25]
$$

### 5. So your question is absolutely valid

You asked:

> "Can't we simply perform the encoding for the variables? Why do we need OVA?"

Yes, we can.

We could directly calculate:

$$
\frac{\#A\text{ in category}}{\#\text{category}},\quad
\frac{\#B\text{ in category}}{\#\text{category}},\quad
\frac{\#C\text{ in category}}{\#\text{category}}
$$

That gives exactly the same class proportions.

OVA is essentially a convenient formulation for obtaining those class-wise probabilities.

It is not some additional mathematical requirement imposed by target encoding.

### 6. Why does scikit-learn use OVA then?

Because the fundamental target-encoding machinery can be expressed in terms of a binary target.

For each class:

```text
Original multiclass target
          ↓
      Class A?
       /    \
     Yes     No
      1       0
          ↓
Target encoding
```

Then repeat for B, C, etc.

This lets the same target-encoding/smoothing machinery be applied to each class.

And importantly, smoothing also happens class-wise.

For example:

$$
TE_A(c) = \lambda_c P(A|c) + (1-\lambda_c)P(A)
$$

$$
TE_B(c) = \lambda_c P(B|c) + (1-\lambda_c)P(B)
$$

and so on.

## 20. Critical Interview Point: fit_transform() vs fit().transform()

With scikit-learn's TargetEncoder, training data should generally be transformed using:

```python
encoder.fit_transform(X_train, y_train)
```

because fit_transform() uses internal cross-fitting.

Conceptually:

```text
Training data
     ↓
Split into K folds
     ↓
Calculate encoding using K-1 folds
     ↓
Transform remaining fold
     ↓
Repeat for every fold
```

This reduces target leakage.

In contrast:

```python
encoder.fit(X_train, y_train)
encoder.transform(X_train)
```

does not use the same cross-fitting scheme and can lead to overfitting. scikit-learn explicitly recommends the cross-fitting approach for training data.

## 21. Complete Mental Model

Remember Target Encoding as a sequence:

```text
Categorical Variable
        ↓
Calculate category target statistic
        ↓
Problem: rare categories are unreliable
        ↓
Introduce Global Mean
        ↓
Need to decide how much to trust
category mean vs global mean
        ↓
Calculate λ based on category size
        ↓
Small n → λ small → trust global mean
Large n → λ large → trust category mean
        ↓
Regularized Target Encoding
```

## 22. Interview Answer in 30 Seconds

Target Encoding converts a categorical variable into a numerical representation based on its relationship with the target. The naive approach uses the category-specific target mean, but this can overfit for rare categories because their statistics are unreliable. Therefore, we shrink the category mean toward the global target mean. The amount of shrinkage depends on the number of observations in that category. category_encoders uses an S-shaped sigmoid weighting controlled by min_samples_leaf and smoothing, while scikit-learn uses a simpler shrinkage factor $n/(n+m)$, with smooth="auto" providing an empirical Bayes estimate of the smoothing parameter. To prevent target leakage during training, scikit-learn's fit_transform() uses internal cross-fitting.

### The one conceptual correction I'd especially remember

Don't think of `min_samples_leaf` as:

> **"The minimum number of samples required before the category mean is used."**

Instead think:

> **"`min_samples_leaf` is the category size at which the sigmoid weight reaches 0.5."**

And don't think of `smoothing` as merely "making the curve smooth." More precisely:

> **Higher `smoothing` makes the S-curve flatter, causing a more gradual transition in trust between the global mean and category mean.**

That distinction will make your explanation much stronger in an interview.


In [1]:
import pandas as pd
from sklearn.preprocessing import TargetEncoder

In [2]:
# Sample data
data = {
    'Feature': ['A', 'B', 'A', 'B', 'C', 'A', 'B', 'C'],
    'Target': [1, 0, 0, 1, 1, 1, 0, 1]
}
df = pd.DataFrame(data)

# Separating the feature and target columns
X = df.drop('Target', axis=1)
y = df['Target']

# Initialize the TargetEncoder
encoder = TargetEncoder(smooth='auto')

# Fit the encoder using the feature data and target variable
encoder.fit(X, y)

# Transform the data
encoded = encoder.transform(X)

encoded

array([[0.65666041],
       [0.40337711],
       [0.65666041],
       [0.40337711],
       [1.        ],
       [0.65666041],
       [0.40337711],
       [1.        ]])